In [ ]:
# Voltage clamp "virtuale" del modello mntb_hh_kv3_neuron (Wang 1998 Fig. 2 e 4)
# V e' imposta: ogni gate segue x(t) = x_inf + (x0 - x_inf) * exp(-t/tau), soluzione esatta.
# Il sodio e' escluso (in Wang c'e' TTX). Parametri = quelli del .nestml.
import numpy as np
import matplotlib.pyplot as plt

# ---- parametri (copiati dal .nestml) ----
g_LT, g_Kv3, E_K = 20., 150., -80.            # nS, nS, mV
K = {  # gate: (k_alpha, eta_alpha, k_beta, eta_beta)
    "l": (1.2, 0.03512, 0.2248, -0.0319),
    "r": (0.0438, -0.0053, 0.0562, -0.0047),
    "n": (0.2719, 0.04, 0.1974, 0.0),
    "p": (0.00713, -0.1942, 0.0935, 0.0058),
}
def rates(g, V):
    ka, ea, kb, eb = K[g]; return ka*np.exp(ea*V), kb*np.exp(eb*V)
def x_inf(g, V):
    a, b = rates(g, V); return a/(a+b)
def x_t(g, x0, V, t):
    a, b = rates(g, V); xi, tau = a/(a+b), 1./(a+b)
    return xi + (x0 - xi)*np.exp(-t/tau)
def steady(V): return {g: x_inf(g, V) for g in K}
def evolve(state, V, t): return {g: x_t(g, state[g], V, t) for g in K}
def currents(s, V, gk=g_Kv3):   # nA, positive = outward (convenzione della figura di Wang)
    I_ht = gk*s["n"]**3*(0.8+0.2*s["p"])*(V-E_K)/1000.
    I_lt = g_LT*s["l"]*s["r"]*(V-E_K)/1000.
    return I_ht, I_lt

tt = np.arange(0, 200.0001, 0.05)
Vs = np.arange(-80, 61, 20)

def protocol(hold, prepulse=None, gk=g_Kv3):
    out = {}
    for V in Vs:
        s = steady(hold)
        if prepulse is not None:
            s = evolve(s, prepulse[0], prepulse[1])
        st = evolve(s, V, tt)
        out[V] = currents(st, V, gk)
    return out

# Protocollo A (Wang 2A): hold -80, prepulse -100 mV 30 ms, step 200 ms
A  = protocol(-80, (-100., 30.))
A_tea = protocol(-80, (-100., 30.), gk=0.0)    # "TEA": Kv3 = 0
# Protocollo C (Wang 2C): hold -40 (gates all'equilibrio a -40)
C  = protocol(-40)

i195 = np.searchsorted(tt, 195.)
def sust(P, V): return P[V][0][i195] + P[V][1][i195]

# ---------------- Figura 1: tracce e I-V ----------------
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for V in Vs:
    ax[0,0].plot(tt, A[V][0]+A[V][1], lw=1)
    ax[0,1].plot(tt, A_tea[V][0]+A_tea[V][1], lw=1)
    ax[0,2].plot(tt, C[V][0]+C[V][1], lw=1)
ax[0,0].set_title("Totale (hold -80 mV)  [Wang 2A sx]"); ax[0,1].set_title("Kv3 = 0 ('TEA')  [Wang 2A dx]")
ax[0,2].set_title("Hold -40 mV  [Wang 2C]")
for a in ax[0]: a.set_xlabel("t (ms)"); a.set_ylabel("I (nA)"); a.grid(alpha=.3)

norm = sust(A, 60)
Vf = np.arange(-80, 61, 2.)
def iv(hold, pre, gk=g_Kv3):
    res = []
    for V in Vf:
        s = steady(hold)
        if pre: s = evolve(s, pre[0], pre[1])
        s = evolve(s, V, 195.)
        res.append(currents(s, V, gk))
    return np.array(res)
ivA = iv(-80, (-100., 30.)); ivA_tea = iv(-80, (-100., 30.), 0.0); ivC = iv(-40, None)
tot, ht, lt = ivA.sum(1), ivA[:,0], ivA[:,1]
ax[1,0].plot(Vf, tot/tot[-1], "k-", label="controllo (hold -80)")
ax[1,0].plot(Vf, ivA_tea.sum(1)/tot[-1], "r-", label="Kv3=0 ('TEA')")
ax[1,0].set_title("I-V totale normalizzata  [Wang 2B]")
ax[1,1].plot(Vf, ivC.sum(1)/tot[-1], "k-", label="hold -40, totale")
ax[1,1].plot(Vf, ivC[:,0]/tot[-1], "b--", label="solo Kv3 (I_HT)")
ax[1,1].plot(Vf, ivC[:,1]/tot[-1], "g--", label="solo I_LT")
ax[1,1].set_title("Hold -40 mV  [Wang 2D]")
ax[1,2].plot(Vf, ht/tot[-1], "b-", label="I_HT (Kv3)"); ax[1,2].plot(Vf, lt/tot[-1], "g-", label="I_LT")
ax[1,2].set_title("I_HT vs I_LT  [Wang 2F]")
for a in ax[1]: a.set_xlabel("V (mV)"); a.set_ylabel("I / I_tot(+60)"); a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("/mnt/user-data/outputs/vc_fig1_IV.png", dpi=120); plt.close()

# ---------------- Rise time 10-90% di I_HT (Wang 4C) ----------------
tfine = np.arange(0, 40, 0.005)
def rise(V, hold=-40):
    s = evolve(steady(hold), V, tfine)
    I = currents(s, V)[0]; Imax = I[-1] if V>=20 else I.max()
    Ipk = I.max(); i10 = np.argmax(I >= 0.1*Ipk); i90 = np.argmax(I >= 0.9*Ipk)
    return tfine[i90]-tfine[i10]
Vr = [0, 20, 40, 60]
rt = [rise(V) for V in Vr]
# ---------------- Tail current (Wang 4D-F): 100 ms a +40, poi -20 mV ----------------
tail_t = np.arange(0, 40, 0.01)
s = evolve(steady(-40), 40., 100.)
st = evolve(s, -20., tail_t)
Itail = currents(st, -20.)[0]
# fit esponenziale su decadimento da picco
j0 = np.argmax(Itail)                 # a -20 mV la corrente e' ancora uscente (E_K = -80): decade da qui
y = Itail[j0:]; tt2 = tail_t[j0:]-tail_t[j0]
from scipy.optimize import curve_fit      # decadimento verso un plateau (n_inf a -20 mV non e' 0)
f = lambda t, A, tau, c: A*np.exp(-t/tau) + c
popt, _ = curve_fit(f, tt2, y, p0=[y[0], 3., y[-1]])
tau_tail = popt[1]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for V in Vr:
    ax[0].plot(tfine, currents(evolve(steady(-40), V, tfine), V)[0], label=f"{V} mV")
ax[0].set_xlim(0, 15); ax[0].set_title("Attivazione Kv3 (hold -40)  [Wang 4A/B]"); ax[0].legend(); ax[0].set_xlabel("t (ms)"); ax[0].set_ylabel("I_Kv3 (nA)")
ax[1].plot(Vr, rt, "ko-", label="modello")
ax[1].plot([0, 60], [5.53, 0.50], "rs", label="Wang (MNTB: 5.53 -> 0.50 ms)")
ax[1].set_title("Rise time 10-90%  [Wang 4C]"); ax[1].legend(); ax[1].set_xlabel("V (mV)"); ax[1].set_ylabel("ms")
ax[2].plot(tail_t, Itail); ax[2].set_title(f"Tail a -20 mV: tau = {tau_tail:.1f} ms (Wang: 5.2 ms)  [4D-F]")
ax[2].set_xlabel("t (ms)")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/mnt/user-data/outputs/vc_fig2_kinetics.png", dpi=120); plt.close()

# ---------------- Numeri di confronto ----------------
frac_lt = lt[-1]/tot[-1]
print("=== Confronto con Wang 1998 ===")
print(f"I_LT / totale a +60 mV (hold -80)  : modello {100*frac_lt:.1f}%   Wang 22.9 +/- 4.4%")
print(f"I_HT / totale a +60 mV             : modello {100*(1-frac_lt):.1f}%   Wang 77.2 +/- 4.3%")
print(f"Totale con Kv3=0 / controllo a +60 : modello {100*ivA_tea.sum(1)[-1]/tot[-1]:.1f}%   (blocco totale)")
print(f"  con blocco del 75% di Kv3 (come TEA 1 mM, Wang) : modello {100*(frac_lt+(1-frac_lt)*0.25):.1f}%   Wang (TEA) 36.1 +/- 6.3%")
print(f"Corrente totale a +60 mV           : modello {tot[-1]:.1f} nA   Wang >15 nA")
print("Rise time 10-90% I_Kv3 (ms):", dict(zip(Vr, np.round(rt,2))), "  Wang: 5.53 (0 mV) -> 0.50 (+60 mV)")
print(f"Tau tail a -20 mV                  : modello {tau_tail:.1f} ms   Wang 5.2 +/- 0.3 ms")
# soglia di attivazione: tensione a cui I supera 2% del valore a +60
def thr(I): return Vf[np.argmax(np.abs(I) > 0.02*np.abs(I[-1]))]
print(f"Soglia attivazione (2% di I+60): I_HT {thr(ht)} mV, I_LT {thr(lt)} mV   Wang: I_LT ~ 20 mV piu' negativa di I_HT, I_HT ~ -20 mV")
print(f"I_LT a -20 mV (hold -80): {lt[np.argmin(abs(Vf+20))]*1000:.0f} pA  Wang (DTX-sensibile): ~0.3-0.5 nA")
